# 🥥 Janjang Vision - Training Model Deteksi Janjang Kelapa Sawit

Notebook ini:
1. Install dependensi
2. Download 3 dataset Roboflow (bunchtest + nazwa + gcstech)
3. Gabungkan semua → 1 kelas "janjang"
4. Training YOLO11
5. Download model hasil

**Butuh:** Roboflow API key (gratis, daftar di roboflow.com)

In [ ]:
# 1) Install dependensi
!pip install ultralytics roboflow -q

In [ ]:
# 2) Masukkan API key Roboflow kamu (dapatkan di https://app.roboflow.com/settings/account)
ROBOFLOW_API_KEY = 'Eyqlt3AThngOY62xeu57'  # ← ganti dengan API key kamu

assert ROBOFLOW_API_KEY != 'MASUKKAN_API_KEY_KAMU_DISINI', 'Ganti API key dulu!'
print('API key OK')

In [ ]:
# 3) Download 3 dataset dari Roboflow Universe
from roboflow import Roboflow
import os, shutil

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

datasets = {}

# Dataset 1: bunchtest (237 gambar, close-up FFB)
print('=== Downloading bunchtest ===')
d1 = rf.workspace('bunchtest-33ooi').project('fresh-fruit-bunch').version(1).download('yolov8', location='ds_bunchtest')
datasets['bunchtest'] = 'ds_bunchtest'

# Dataset 2: nazwa (2.400+ gambar, scene kebun)
print('=== Downloading nazwa ===')
d2 = rf.workspace('nazwa').project('palm-oil-fruit-bunch-k').version(4).download('yolov8', location='ds_nazwa')
datasets['nazwa'] = 'ds_nazwa'

# Dataset 3: gcstech (2.700+ gambar, 5 kelas ripeness)
print('=== Downloading gcstech ===')
d3 = rf.workspace('gcstech').project('oil-palm-fruit-bunch-vlynl').version(2).download('yolov11', location='ds_gcstech')
datasets['gcstech'] = 'ds_gcstech'

print('\nSemua dataset berhasil diunduh!')
for name, path in datasets.items():
    for sp in ['train', 'valid', 'test']:
        d = os.path.join(path, sp, 'images')
        if os.path.isdir(d):
            print(f'  {name}/{sp}: {len(os.listdir(d))} images')

In [ ]:
# 4) Gabungkan 3 dataset → 1 kelas "janjang" (class 0)
import os
import glob
import shutil
from pathlib import Path

OUT = Path('merged')
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

SPLITS = ['train', 'valid', 'test']
ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / 'merge_dataset',
]

DS_LIST = []
seen = set()
for root in ROOT_CANDIDATES:
    if not root.exists():
        continue
    for ds_name in ['ds_bunchtest', 'ds_nazwa', 'ds_gcstech']:
        ds_path = root / ds_name
        if ds_path.exists() and str(ds_path) not in seen:
            seen.add(str(ds_path))
            DS_LIST.append(ds_path)

if not DS_LIST:
    raise FileNotFoundError('Dataset Roboflow tidak ditemukan. Pastikan folder ds_bunchtest, ds_nazwa, ds_gcstech ada di root project atau di subfolder merge_dataset/.')

stats = {}
for split in SPLITS:
    img_out = OUT / split / 'images'
    lbl_out = OUT / split / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    count = 0

    for ds in DS_LIST:
        img_dir = ds / split / 'images'
        lbl_dir = ds / split / 'labels'
        if not img_dir.is_dir():
            continue

        for img_file in sorted(img_dir.glob('*.*')):
            stem = img_file.stem
            lbl_file = lbl_dir / f'{stem}.txt'
            if not lbl_file.exists():
                continue

            try:
                with open(lbl_file, 'r', encoding='utf-8') as f:
                    lines = [line.strip() for line in f if line.strip()]
            except Exception:
                continue

            if not lines:
                continue

            new_lines = []
            for line in lines:
                parts = line.split()
                if len(parts) >= 5:
                    parts[0] = '0'  # remap semua kelas ke 0 (janjang)
                    new_lines.append(' '.join(parts))

            if not new_lines:
                continue

            dest_img = img_out / f'{ds.name}_{img_file.name}'
            dest_lbl = lbl_out / f'{ds.name}_{stem}.txt'
            shutil.copy2(img_file, dest_img)
            with open(dest_lbl, 'w', encoding='utf-8') as f:
                f.write('\n'.join(new_lines))
            count += 1

    stats[split] = count

base_path = os.path.abspath(str(OUT))
yaml_content = f'''path: {base_path}
train: train/images
val: valid/images
test: test/images

nc: 1
names: ['janjang']
'''
(OUT / 'data.yaml').write_text(yaml_content, encoding='utf-8')

print('Dataset gabungan siap!')
print(f'  train: {stats["train"]} gambar')
print(f'  valid: {stats["valid"]} gambar')
print(f'  test:  {stats["test"]} gambar')
print('\ndata.yaml:')
print((OUT / 'data.yaml').read_text(encoding='utf-8'))


In [ ]:
%%writefile janjang_counter.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import argparse, math, sys
from pathlib import Path

try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

try:
    from ultralytics import YOLO
except ImportError:
    sys.exit("ERROR: pip install ultralytics opencv-python")

try:
    import cv2
except ImportError:
    sys.exit("ERROR: pip install opencv-python")

import numpy as np

JANJANG_NAMES = {"janjang", "tbs", "ffb", "bunch", "fruit_bunch", "fruit"}

def cari_kelas_janjang(names):
    for cid, nama in names.items():
        if str(nama).lower() in JANJANG_NAMES:
            return cid
    return None

def deteksi_frame(model, frame_bgr, conf, classes):
    results = model.predict(source=frame_bgr, conf=conf, classes=classes, verbose=False)
    r = results[0]
    names = r.names
    cls_ids = r.boxes.cls.int().tolist() if r.boxes is not None else []
    per_kelas = {}
    for c in cls_ids:
        nama = names.get(c, str(c))
        per_kelas[nama] = per_kelas.get(nama, 0) + 1
    return per_kelas, r.plot()

def hitung_gambar(model, path_foto, conf, classes):
    img = cv2.imread(str(path_foto))
    if img is None:
        sys.exit(f"ERROR: tidak bisa membaca gambar: {path_foto}")
    per_kelas, annotated = deteksi_frame(model, img, conf, classes)
    path_hasil = path_foto.with_name(path_foto.stem + "_hasil" + path_foto.suffix)
    cv2.imwrite(str(path_hasil), annotated)
    return per_kelas, path_hasil

def cetak_ringkasan(per_kelas, path_hasil=None):
    total = sum(per_kelas.values())
    print("=" * 52)
    print("HASIL DETEKSI")
    print("=" * 52)
    if not per_kelas:
        print("  (tidak ada objek terdeteksi)")
    for nama, n in per_kelas.items():
        print(f"  {nama:<20}: {n}")
    print("-" * 52)
    print(f"  TOTAL            : {total}")
    if path_hasil:
        print(f"  Gambar hasil     : {path_hasil}")
    print("=" * 52)
    return total

def mode_foto(model, path_foto, conf, classes):
    per_kelas, path_hasil = hitung_gambar(model, path_foto, conf, classes)
    cetak_ringkasan(per_kelas, path_hasil)

def _mode_stream(model, cap, conf, classes, save, path_video=None):
    if not cap.isOpened():
        sys.exit("ERROR: tidak bisa membuka sumber video/kamera.")
    writer = None
    out_path = None
    if save and path_video is not None:
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out_path = path_video.with_name(path_video.stem + "_hasil.mp4")
        writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    print("Tekan 'q' untuk keluar.")
    total_akumulasi = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        per_kelas, annotated = deteksi_frame(model, frame, conf, classes)
        total = sum(per_kelas.values())
        total_akumulasi += total
        overlay = annotated.copy()
        teks = f"Janjang: {total}   (akumulasi: {total_akumulasi})"
        cv2.putText(overlay, teks, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)
        cv2.imshow("Janjang Vision - tekan 'q' untuk keluar", overlay)
        if writer is not None:
            writer.write(overlay)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    if writer is not None:
        writer.release()
    cv2.destroyAllWindows()
    print(f"Selesai. Total janjang terdeteksi: {total_akumulasi}")
    if out_path:
        print(f"Video hasil disimpan: {out_path}")

def mode_video(model, path_video, conf, classes, save):
    cap = cv2.VideoCapture(str(path_video))
    _mode_stream(model, cap, conf, classes, save, path_video)

def mode_kamera(model, conf, classes):
    cap = cv2.VideoCapture(0)
    _mode_stream(model, cap, conf, classes, save=False)

def mode_train(dataset_yaml, epochs, imgsz, batch, patience):
    if not Path(dataset_yaml).exists():
        sys.exit(f"ERROR: file dataset tidak ditemukan: {dataset_yaml}")
    model = YOLO("yolo11n.pt")
    print(f"Mulai training {epochs} epoch pada dataset: {dataset_yaml}")
    model.train(data=dataset_yaml, epochs=epochs, imgsz=imgsz,
                batch=batch, patience=patience, name="janjang_train")
    best_path = Path(model.trainer.save_dir) / "weights" / "best.pt"
    print("=" * 52)
    print("Training selesai!")
    print(f"Model terbaik: {best_path}")
    print(f"Pakai dengan:  python janjang_counter.py foto.jpg --model {best_path}")
    print("=" * 52)

def buat_gambar_demo(path_out="demo_janjang.jpg"):
    rng = np.random.default_rng(42)
    h, w = 720, 960
    img = np.full((h, w, 3), (55, 130, 60), dtype=np.uint8)
    for _ in range(12):
        cx = int(rng.integers(90, w - 90))
        cy = int(rng.integers(90, h - 90))
        r = int(rng.integers(30, 50))
        cv2.circle(img, (cx, cy), r, (38, 55, 85), -1)
        cv2.circle(img, (cx, cy), int(r * 0.6), (60, 80, 110), -1)
        for _ in range(12):
            ang = rng.uniform(0, 2 * math.pi)
            x1 = int(cx + rng.uniform(-r * 0.5, r * 0.5))
            y1 = int(cy + rng.uniform(-r * 0.5, r * 0.5))
            x2 = int(x1 + 14 * math.cos(ang))
            y2 = int(y1 + 14 * math.sin(ang))
            cv2.line(img, (x1, y1), (x2, y2), (25, 40, 65), 3)
    cv2.imwrite(path_out, img)
    print(f"Gambar demo dibuat: {path_out}")
    return Path(path_out)

def main():
    p = argparse.ArgumentParser(description="Janjang Vision - hitung janjang kelapa sawit (TBS) dengan AI (YOLO).")
    p.add_argument("source", nargs="?", help="path foto")
    p.add_argument("--model", default="yolo11n.pt")
    p.add_argument("--conf", type=float, default=0.35)
    p.add_argument("--classes", help="filter kelas, contoh: 0,1")
    p.add_argument("--video", help="path video/CCTV")
    p.add_argument("--cam", action="store_true")
    p.add_argument("--save", action="store_true")
    p.add_argument("--train", help="path dataset.yaml")
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--batch", type=int, default=-1)
    p.add_argument("--patience", type=int, default=20)
    p.add_argument("--demo", action="store_true")
    args = p.parse_args()

    if args.train:
        mode_train(args.train, args.epochs, args.imgsz, args.batch, args.patience)
        return

    print(f"Memuat model: {args.model} ...")
    model = YOLO(args.model)
    names = model.names

    classes = None
    if args.classes:
        classes = [int(x) for x in args.classes.split(",")]
    else:
        cid = cari_kelas_janjang(names)
        if cid is not None:
            classes = [cid]
            print(f"Model mengenali kelas janjang ('{names[cid]}') - hanya kelas ini yang dihitung.")
        else:
            print("Model tidak punya kelas janjang spesifik - menghitung semua objek terdeteksi.")

    if args.demo:
        path_demo = buat_gambar_demo()
        per_kelas, path_hasil = hitung_gambar(model, path_demo, args.conf, classes)
        cetak_ringkasan(per_kelas, path_hasil)
        return

    if args.video:
        mode_video(model, Path(args.video), args.conf, classes, args.save)
    elif args.cam:
        mode_kamera(model, args.conf, classes)
    elif args.source:
        mode_foto(model, Path(args.source), args.conf, classes)
    else:
        p.print_help()

if __name__ == "__main__":
    main()


In [ ]:
# 6) Training!
!python janjang_counter.py --train merged/data.yaml --epochs 100 --imgsz 640 --batch 16 --patience 25

In [ ]:
# 7) Download model hasil
import os
import glob
import shutil
from pathlib import Path

models = sorted(glob.glob('runs/detect/*/weights/best.pt'))
if models:
    best_model = models[-1]
    print('Model ditemukan:', best_model)

    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle').exists():
        from IPython.display import FileLink, display
        print('Environment: Kaggle')
        print('Gunakan link di bawah untuk download model:')
        display(FileLink(best_model))
    else:
        try:
            from google.colab import files
            files.download(best_model)
            print('File berhasil didownload via Colab.')
        except Exception:
            target = os.path.join(os.getcwd(), os.path.basename(best_model))
            shutil.copy2(best_model, target)
            print(f'File disalin ke lokasi lokal: {target}')
            print('Environment ini bukan Colab, jadi file disalin ke folder kerja lokal.')
else:
    print('Model belum ada - training mungkin belum selesai')


## Setelah dapat best.pt

Taruh file `best.pt` di folder project kamu, lalu jalankan:
```
python janjang_counter.py foto.jpg --model best.pt
```